# Export to Google Spreadsheet 
Using gspread

Ref: 
1. https://docs.gspread.org/en/
2. Authorization: https://docs.gspread.org/en/v6.1.4/oauth2.html#enable-api-access 

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [36]:
import gspread
from gspread_formatting import *
from google.oauth2.service_account import Credentials
from gspread.exceptions import WorksheetNotFound

from utils import datetime_generator

## Google API Authorization 

In [3]:
SCOPES = [
    'https://www.googleapis.com/auth/spreadsheets',
    'https://www.googleapis.com/auth/drive'
]

credentials = Credentials.from_service_account_file(
    "../credentials.json",
    scopes=SCOPES
)

In [4]:
gc = gspread.authorize(credentials)

In [5]:
if gc:
    print("Gspread authorized")
else:
    print("Gspread failed to authorize")

Gspread authorized


## Test Data

In [6]:
from typing import Optional, List, Dict
from dataclasses import dataclass

@dataclass
class Job:
    title: str
    url: str
    text: Optional[str]
    searched_via: str
    low_quality: bool = False
    keep: Optional[bool] = None
    score: Optional[int] = None
    reason: str = ""
    is_ai_role: Optional[bool] = None
    manual_check_required: bool = False

In [7]:
import test_data
import test_manual_check_jobs

test_valid_data = test_data.test_data
test_mc_data = test_manual_check_jobs.test_manual_check_data

In [8]:
len(test_valid_data)

5

In [9]:
len(test_mc_data)

5

## Create Spreadsheet

In [10]:
job_sheet = gc.open_by_key("1CQioaeiozCNXEMts5e6T1_Ls4Wcc8qVJ4YbbhAAWQwE")

In [11]:
if job_sheet:
    print(f"'{job_sheet.title}' has been loaded successfully!")
else:
    print("Failed to load the spreadsheet")

'remote_jobs_ai_engineering' has been loaded successfully!


In [14]:
### Create a new worksheet unless the same name based on the datetime doesn't exist

sheet_title = f"{datetime_generator.generate_current_datetime()}_VALID"

try: 
    worksheet_valid = job_sheet.worksheet(sheet_title)
    print("Worksheet already exists")
except WorksheetNotFound:
    worksheet_valid = job_sheet.add_worksheet(
        title=sheet_title, 
        rows=500,
        cols=10
    )

Worksheet already exists


In [ ]:
val = worksheet.acell('B1').value

In [38]:
print(val)

None


In [39]:
val2 = worksheet.cell(1,2).value

print(val2)

Keep


In [40]:
values_list = worksheet.row_values(1)
print(values_list)

['Title', 'Keep', 'Score', 'Reason', 'URL', 'Source']


In [41]:
col_value_list = worksheet.col_values(2)

print(col_value_list)

['Keep', 'TRUE', 'TRUE', 'TRUE', 'TRUE', 'TRUE']


In [42]:
worksheet.update_acell("B1", "Bingo!")

{'spreadsheetId': '1CQioaeiozCNXEMts5e6T1_Ls4Wcc8qVJ4YbbhAAWQwE',
 'updatedRange': "'2026-04-18'!B1",
 'updatedRows': 1,
 'updatedColumns': 1,
 'updatedCells': 1}

## Export Valid Jobs to Spreadsheet

In [49]:
col_headers = ["Title", "Keep", "Score", "Reason", "URL", "Source"]
col_headers_len = len(col_headers)

def set_header_range():
    
    return {
    1: "A",
    2: "B",
    3: "C",
    4: "D",
    5: "E",
    6: "F",
    7: "G",
    8: "H",
    9: "I",
    10: "J"
}
    
rows = [
    col_headers,
    *[
        [job.title, job.keep, job.score, job.reason, job.url, job.searched_via]
        for job in test_valid_data
    ]
]

#### Header ####
worksheet_valid.format(
    ranges=f"A1:{set_header_range()[col_headers_len]}",
    format={
        "backgroundColor": {
            "red": 0.72,
            "green": 0.84,
            "blue": 0.96
        },
        "horizontalAlignment": "CENTER",
        "textFormat": {
            "bold": True
        }
    }
)

#### Rows ####
worksheet_valid.format(
    ranges=f"A2:{set_header_range()[col_headers_len]}",
    format={
        "backgroundColor": {
            "red": 0.92,
            "green": 0.96,
            "blue": 1.0
        },
        "horizontalAlignment": "LEFT",
        "textFormat": {
            "bold": False
        }
    }
)

#### Width ####
set_column_widths(worksheet_valid, [
    ('A', 300), ('B', 50), ('C', 50), ('D', 300), ("E", 150), ("F", 50)
])

worksheet_valid.update("A1", rows)

/var/folders/y7/jzgx63gj4rb9p1g3z163h7l80000gn/T/ipykernel_4764/396119563.py:65: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  worksheet_valid.update("A1", rows)


{'spreadsheetId': '1CQioaeiozCNXEMts5e6T1_Ls4Wcc8qVJ4YbbhAAWQwE',
 'updatedRange': "'2026-04-18_VALID'!A1:F6",
 'updatedRows': 6,
 'updatedColumns': 6,
 'updatedCells': 36}

## Export Manual-Check-Required Jobs to Spreadsheet

In [18]:
### Create a new worksheet unless the same name based on the datetime doesn't exist

sheet_title = f"{datetime_generator.generate_current_datetime()}_Manual Check"

try: 
    worksheet_mcr = job_sheet.worksheet(sheet_title)
    print("Worksheet already exists")
except WorksheetNotFound:
    worksheet_mcr = job_sheet.add_worksheet(
        title=sheet_title, 
        rows=500,
        cols=10
    )

Worksheet already exists


In [51]:
mcr_col_headers = ["Title", "Keep", "Score", "Reason", "URL", "Source", "MCR"]
mcr_col_headers_len = len(mcr_col_headers)


rows = [
    mcr_col_headers,
    *[
        [job.title, job.keep, job.score, job.reason, job.url, job.searched_via, job.manual_check_required]
        for job in test_mc_data
    ]
]

#### Header ####
worksheet_mcr.format(
    ranges=f"A1:{set_header_range()[mcr_col_headers_len]}",
    format={
        "backgroundColor": {
            "red": 0.72,
            "green": 0.84,
            "blue": 0.96
        },
        "horizontalAlignment": "CENTER",
        "textFormat": {
            "bold": True
        }
    }
)

#### Rows ####
worksheet_mcr.format(
    ranges=f"A2:{set_header_range()[mcr_col_headers_len]}",
    format={
        "backgroundColor": {
            "red": 0.92,
            "green": 0.96,
            "blue": 1.0
        },
        "horizontalAlignment": "LEFT",
        "textFormat": {
            "bold": False
        }
    }
)

#### Width ####
set_column_widths(worksheet_mcr, [
    ('A', 300), ('B', 50), ('C', 50), ('D', 300), ("E", 150), ("F", 50), ("G", 50)
])


worksheet_mcr.update("A1", rows)

/var/folders/y7/jzgx63gj4rb9p1g3z163h7l80000gn/T/ipykernel_4764/2791227291.py:51: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  worksheet_mcr.update("A1", rows)


{'spreadsheetId': '1CQioaeiozCNXEMts5e6T1_Ls4Wcc8qVJ4YbbhAAWQwE',
 'updatedRange': "'2026-04-18_Manual Check'!A1:G6",
 'updatedRows': 6,
 'updatedColumns': 7,
 'updatedCells': 42}